## Step 1 — Load the clean data and split it

**Why:** we start from the saved clean file, so we don't redo all the data preparation. We must get **exactly the same split** as in the data preparation notebook (same rows in train and test). Otherwise the test set would not be "unseen" anymore.

### 1a — Load

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

df = pd.read_csv('diabetes_clean_2015.csv')
print(df.shape)
df.head()

(440629, 30)


,GenHealth,PhysHealthDays,MentHealthDays,DiffWalk,HighBP,HighChol,CholCheck,Stroke,HeartDisease,KidneyDisease,COPD,Arthritis,Depression,Asthma,Smoker,PhysActivity,HeavyDrinker,Fruits,Veggies,HealthPlan,NoDocBcCost,Male,AgeGroup,Education,Income,Race,Marital,DIABETE3,target,BMI
0,5.0,15.0,18.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,3.0,0.0,0.0,0.0,1.0,1.0,0.0,0,9.0,2.0,2.0,1.0,1.0,3.0,0,40.18
1,3.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0,7.0,4.0,1.0,1.0,2.0,3.0,0,25.09
2,4.0,15.0,0.0,NaN,0.0,1.0,1.0,1.0,NaN,0.0,0.0,1.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,1.0,0.0,0,11.0,2.0,NaN,1.0,2.0,3.0,0,22.04
3,5.0,30.0,30.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,4.0,0.0,0.0,1.0,0.0,1.0,1.0,0,9.0,2.0,5.0,1.0,1.0,3.0,0,28.19
4,5.0,20.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,4.0,0.0,0.0,NaN,1.0,1.0,0.0,0,9.0,3.0,NaN,1.0,1.0,3.0,0,24.37


In [2]:
# quick check: the file is the one i saved at the end of data preparation
print('missing values per column (top 5):')
print(df.isna().sum().sort_values(ascending=False).head())
print()
print(df['target'].value_counts().sort_index())   # 0 = No, 1 = Pre-diabetes, 2 = Diabetes

missing values per column (top 5):
Income          79143
HighChol        62397
Veggies         50893
Fruits          43502
PhysActivity    37823
dtype: int64

target
0    375686
1      7690
2     57253
Name: count, dtype: int64


### 1b — Features (X), target (y) and feature types

The feature types are the same as in Step 6 of the data preparation (with the readable names from Step 10).

In [5]:
numeric_cols = ['BMI', 'PhysHealthDays', 'MentHealthDays']
ordinal_cols = ['GenHealth', 'AgeGroup', 'Education', 'Income', 'Smoker', 'CholCheck']
nominal_cols = ['Race', 'Marital']
binary_cols  = ['DiffWalk', 'HighBP', 'HighChol', 'Stroke', 'HeartDisease', 'KidneyDisease', 'COPD',
                'Arthritis', 'Depression', 'Asthma', 'PhysActivity', 'HeavyDrinker', 'Fruits', 'Veggies',
                'HealthPlan', 'NoDocBcCost', 'Male']

features = numeric_cols + ordinal_cols + nominal_cols + binary_cols

X = df[features]
y = df['target']

# safety checks: every column has a type, and the target is not in X
assert len(features) == 28 and len(set(features)) == 28
assert set(features) == set(df.columns) - {'DIABETE3', 'target'}
print('X:', X.shape, ' y:', y.shape)

X: (440629, 28)  y: (440629,)


### 1c — Train / test split

Same settings as Step 11c of the data preparation:
- `test_size=0.2`: 80% train, 20% test
- `stratify=y`: train and test keep the same class proportions (very important with only ~1.7% pre-diabetes)
- `random_state=42`: the same rows are chosen every time

The split only depends on the **number of rows, their order, `y` and these settings**, not on the column order. The CSV keeps the rows in the same order as in data preparation, so we get exactly the same split.

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print('train:', X_train.shape, ' test:', X_test.shape)
pd.DataFrame({'train': y_train.value_counts(normalize=True),
              'test':  y_test.value_counts(normalize=True)}).sort_index().round(3)

train: (352503, 28)  test: (88126, 28)


,train,test
target,,
0,0.853,0.853
1,0.017,0.017
2,0.130,0.130


In [7]:
# same sizes as in data_preparation_2015.ipynb (Step 11c)
assert X_train.shape == (352503, 28) and X_test.shape == (88126, 28)
print('Same split as in data preparation ✓')

Same split as in data preparation ✓


### 1d — Preprocessing pipeline

| Group | Columns | What happens |
|---|---|---|
| `num` | `BMI`, `PhysHealthDays`, `MentHealthDays` | fill `NaN` with the **median**, then **scale** (mean 0, std 1) |
| `ord` | `GenHealth`, `AgeGroup`, `Education`, `Income`, `Smoker`, `CholCheck` | fill `NaN` with the **most frequent** value, keep the order |
| `bin` | 17 yes/no columns | fill `NaN` with the **most frequent** value |
| `nom` | `Race`, `Marital` | fill `NaN`, then **one-hot** encoding (one 0/1 column per category) |

**Important:** these steps **learn** from the data (medians, means, categories). So:
- they are learned on `X_train` only (`fit_transform`), then applied to `X_test` (`transform`)
- in the next steps, `preprocess` goes **inside** a `Pipeline` with the model, so cross-validation also learns it on the training folds only

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocess = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                      ('scale',  StandardScaler())]), numeric_cols),
    ('ord', SimpleImputer(strategy='most_frequent'), ordinal_cols),
    ('bin', SimpleImputer(strategy='most_frequent'), binary_cols),
    ('nom', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                      ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), nominal_cols),
])
preprocess

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('ord', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_n

### 1e — Check the preprocessing

In [9]:
X_train_prep = preprocess.fit_transform(X_train)   # learn on train
X_test_prep  = preprocess.transform(X_test)        # only apply on test

print('train:', X_train_prep.shape, ' test:', X_test_prep.shape)
print('missing values left:', np.isnan(X_train_prep).sum(), np.isnan(X_test_prep).sum())

# names of the columns after preprocessing (Race and Marital become several 0/1 columns)
feature_names = preprocess.get_feature_names_out()
print(len(feature_names), 'columns')
list(feature_names)

train: (352503, 37)  test: (88126, 37)
missing values left: 0 0
37 columns


['num__BMI',
 'num__PhysHealthDays',
 'num__MentHealthDays',
 'ord__GenHealth',
 'ord__AgeGroup',
 'ord__Education',
 'ord__Income',
 'ord__Smoker',
 'ord__CholCheck',
 'bin__DiffWalk',
 'bin__HighBP',
 'bin__HighChol',
 'bin__Stroke',
 'bin__HeartDisease',
 'bin__KidneyDisease',
 'bin__COPD',
 'bin__Arthritis',
 'bin__Depression',
 'bin__Asthma',
 'bin__PhysActivity',
 'bin__HeavyDrinker',
 'bin__Fruits',
 'bin__Veggies',
 'bin__HealthPlan',
 'bin__NoDocBcCost',
 'bin__Male',
 'nom__Race_1.0',
 'nom__Race_2.0',
 'nom__Race_3.0',
 'nom__Race_4.0',
 'nom__Race_5.0',
 'nom__Marital_1.0',
 'nom__Marital_2.0',
 'nom__Marital_3.0',
 'nom__Marital_4.0',
 'nom__Marital_5.0',
 'nom__Marital_6.0']

## Summary of Step 1

| What | Value |
|---|---|
| Clean dataset | 440,629 rows × 28 features |
| Train / test | 352,503 / 88,126 (80 / 20, stratified, `random_state=42`) |
| Classes | ≈ 85.3% No, 1.7% Pre-diabetes, 13.0% Diabetes (same in train and test) |
| After `preprocess` | 37 columns, 0 missing values |